<div style="text-align: center;" >
<h1 style="margin-top: 0.2em; margin-bottom: 0.1em;">Introduction to Computation for the Social Sciences</h1>
<h2 style="margin-top: 0.7em; margin-bottom: 0.3em;">Assignment 2</h2>
<h3 style="margin-top: 0.7em; margin-bottom: 0.3em;">Deadline: Dec 1, 23:59</h3>

<h2 style="margin-top: 0.7em; margin-bottom: 0.3em; color: red"> A Possible Solution </h2>

</div>
<br>

<h4 style="margin-top: 0.7em; margin-bottom: 0.3em; font-style:italic">
Please push your solutions to your personal repository in our <a href='https://classroom.github.com/a/tGD_7t85'>GitHub Classroom</a></h4><br>

***

In this assignment, we want to investigate potential network-structures among the Eurovision Song Contest.

The Eurovision Song Contest is a yearly song contest between european countries. 
Each country can send one representative, performing a music act. 
The winner is determined by a voting system, where a countries inhabitants can vote for *other* countries musicians. 
Based on the number of calls from a certain country to a certain country, points (ranging between 0 and 12) are awarded.

These final points determine the winner of the current year.

<h1>Part 1 - Scrape Eurovision Data from Wikipedia</h1>


<h3>Task 1 - Scrape the years 2010 - 2013 using Beautiful Soup</h3>

In this task, you are supposed to scrape the results from the Eurovision Song Contest for the following years: [2010](https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2010), [2011](https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2011), [2012](https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2012), [2013](https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2013). We selected these years, because for them the html-code is (mostly) of the same structure. This way, you can reuse your code. 

- We are interested in the assigned points in the final, which can be found in the table named something like *'Detailed voting results of the final'*. (be careful, this name might be slightly different for one or more of the years. Please write your code to match all options)
- Try to write the code for scraping in a *convenient way* (meaning try not to copy and paste stuff for each year...)


***Important***: We want you to scrape the resulting tables by using Beautiful Soup and html-tags and html-attributes. Please **do not use** the `pd.read_html()` function.


***a) Investigate the underlying html structure of the wikipedia pages. Where can you find the table of interest? How could you filter for it?***

Access and request the wikipedia page. Deal with the response in a suitable way and filter for the html-code, reffering to our table of interest.

*Hint:* One way to select the desired table is to use the tables caption and the `.parent` function.

In [ ]:
# packages
from bs4 import BeautifulSoup
import requests
import re # for the regex
import pandas as pd
from urllib.parse import quote
from datetime import datetime
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch


In [ ]:
# Scrape and filter for the table

# Define the URL to access - first only for 2010
url = 'https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2010'

# Sending Request
response = requests.get(url)

# Was the request successful? - yes
print(response)

# parsing the data into a better readable format
soup = BeautifulSoup(response.content, 'html.parser')
print(soup.prettify())

# store all captions from the page in a new variable
all_captions = soup.find_all('caption')

In [ ]:
# filter data

table = []

# iterate over all the captions to find the caption 'Detailed voting results of the final'
for caption in all_captions:

    # store current caption in a variable and only show text and strip all the whitespaces
    temp = caption.text.strip()

    # further with the help of a regex - remove all occurences of the citations (Number in square brackets)
    temp = re.sub('(\[[1-9])\w\]+', '', temp)

    # check for the caption 'Detailed voting results of the final'
    if (temp == "Detailed voting results of the final"):
        # if the caption is found store the table associated with the caption in the prior defined variable
        table = caption.parent

***b) Scrape the content of the table***

Once you restricted the response to the code of the table, start filtering out it's content. 

*Hint:* It might make sense to scrape the table using single lists/vectors for certain information and later recombine these into a data frame of the right format. 

In [ ]:
# Scrape for one year

# get column names with list comprehension
columns = [x.text for x in table.find_all('span')]

# loop for contestants with list comprehension
Contestant = [x.text.strip() for x in table.find_all('th', {'scope': 'row'})]

# get a matrix of all rows (no col headers, no contestants)
all_rows = table.find_all('th', {'scope':'row'})

In [ ]:
# there are probably less complicated solutions for this subtask...

# need to initialize list to store the final data in + initialize running index (i) for the loop
df_list = []
i = 0

# Loop over each row and inside each loop a loop over each cell in the row 
while i < len(all_rows):
    # find the next row to the current one (the current one in the beginning is the column names row) and stores it in the variable col
    col = all_rows[i].findNext()
    
    # initialze a list to store all the data from the row
    in_row = []

    # loops over each cell in a row (stops when in_row has the desired length)
    while len(in_row) < len(columns):
        # finds a cell and remove excess info
        cell = col.text.strip()
        # add cell to th in_row list
        in_row.append(cell) 
        # if the cell contains 12 jumps over the next row (since 12s are bold and therefore occupy two 'html units')
        if(cell == "12"):
            # here is where the jumping over happens
            col = col.findNext()
        # go to the next cell
        col = col.findNext()
    
    # once all cells in a row are found, appends the row to a list as one element
    df_list.append(in_row)
    # goes to the beginning of the next row
    col = col.findNext()
    # increases the running index (so that the loop stops once all rows have been properly stored in df_list - see stopping condition on the outer loop)
    i = i+1


In [ ]:
# turn list into a pandas dataframe with the prior extracted columns (points giving countrys) as columns
df_table = pd.DataFrame(df_list, columns= columns)

In [ ]:
# add a column with Contestants (point receiving countries)
df_table.insert(0, "Constestant", Contestant)
df_table

***c) Reuse your code conveniently to scrape not only one year, but the four years (2010 - 2013) into single data frames.***

If you'd like to, you can additionally export the data frames as separate csv-files.

In [ ]:
# create Function 
# this function is created by taking all the steps described above wrapping them in a function and parameterize the URL and the Name of the table

def get_table(url, TableName):

    """
    This function has been build in to read the "Detailed voting results of the final" of the ESC finals in the years 2010-2013 from their respective wikipedia pages.
    To do so it scrapes the data from the url specified (wikipedia page of the ESC year of interest), searches for a table with the specified name and reads the data found in the table into a pandas dataframe.
    
    Parameters
    ----------
    url: String
        the url to the Wikipedia page of a ESC between 2010-2013
        
    TableName: String
       Name of the table on the specified ESC Year Wikipedia page
       (Only designed to work for the table "Detailed voting results of the final" or its equivalent. It is not meant to be used on other tables)
    
    Returns
    -------
    Pandas Dataframe
        A Dataframe containing the information of the table in the url specified as the arguments of this function.
    """

    # scrape and parse the whole Wikipedia page 
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Extract all captions (to find the one we are looking for)
    all_captions = soup.find_all('caption')

    # initialize a table to store the html code of the table once found
    table = []

    # Loop over all captions and compare to the title given as an argument in the function call and store once the title has been found the html code of the table in the table-list initialized above
    for caption in all_captions:
        temp = caption.text.strip()
        temp = re.sub('(\[[1-9])\w\]+', '', temp)
        if (temp == TableName):
            table = caption.parent

    # get all columns of the table
    columns = [x.text for x in table.find_all('span')]
    # get the first (non-numeric) column of the table
    Contestant = [x.text.strip() for x in table.find_all('th', {'scope': 'row'})]
    # get the rest of the table (only the numbers in the table) and store in all_rows
    all_rows = table.find_all('th', {'scope':'row'})

    # initialize the list, where the data from the table html code shall be stored row by row
    df_list = []
    # initialize a running index to stop our flow once all rows have been added to the list
    i = 0
    
    # Double While-loop
    # Loops over each row and in each row is a second loop over each cell in the current row
    # more details see above
    while i < len(all_rows):
        col = all_rows[i].findNext()
        in_row = []
        while len(in_row) < len(columns):
            cell = col.text.strip()
            in_row.append(cell) 
            if(cell == "12"):
                col = col.findNext()
            col = col.findNext()
        df_list.append(in_row)
        col = col.findNext()
        i = i+1
    # stores everything in a pandas df    
    df_table = pd.DataFrame(df_list, columns= columns)
    df_table.insert(0, "Constestant", Contestant)
    
    # returns the wikipedia table as a pandas df
    return df_table

In [ ]:
# apply function
ev_2010 = get_table("https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2010" ,"Detailed voting results of the final") #test
ev_2011 = get_table("https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2011" ,"Detailed voting results of the final")
ev_2012 = get_table("https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2012" ,"Detailed voting results of the final")
ev_2013 = get_table("https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2013" ,"Final voting results")

<h3>Task 2 - Scrape information on each years aritsts using the 'read_html()' function.</h3>

Now, you may use `pd.read_html()` to retreive the tables containing information on each countries artist, their song etc. These tables are called "Participants of the Eurovision Song Contest 20xx"

Store them in separate data frames and make sure to modify them into suitable data types and format.

In [1]:
# use the 'pd.read_html()' function and indexing to get the desired table from the specified Wikipedia page
part_10 = pd.read_html('https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2010')[1]
part_11 = pd.read_html('https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2011')[2]
part_12 = pd.read_html('https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2012')[1]
part_13 = pd.read_html('https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2013')[2]

# Since a loop would be prettier - here is a loop written by a student - can be adapted to fit this code
"""
artist_tables = []
for year in years:
    table = pd.read_html(f'https://en.wikipedia.org/wiki/Eurovision_Song_Contest_{year}', match = f'Participants of the Eurovision Song Contest {year}')
    artist_tables.append(table[0])
    
# Iterate through the names and final_tables to write each DataFrame to a .csv   
names = ['eurovision_artists_2010', 'eurovision_artists_2011', 'eurovision_artists_2012', 'eurovision_artists_2013']
for name, artist_df in zip(names, artist_tables):
    artist_df.to_csv(f"{name}.csv", index=False)  # Save to CSV
    print(f"Saved {name}.csv")
"""

***

<h1>Part 2 - Get data from an API</h1>



To extend our analysis, we would now like you to retreive data from the Wikipedia API.

For every artist competing in one year (you can choose one year between 2010-2013 yourself), we want to get the date, their wikipedia page was created. Using these dates we want to explore the hypothesis that the earlier an artist's wikipedia page was created - i.e. the more popular he/she already was prior to the Contest - the more points the country got.

<h3>Task 3 - Retreive Data from the <a href='https://xtools.wmcloud.org/api#/'>XTools Wikipedia-API</a></h3>

***a) Access the API via the correct `GET request` and retrieve the creation date of each artist's wikipedia page.***

You can use the sandbox available at https://xtools.wmcloud.org/api#/ for testing which GET request you need and how it works.

In [ ]:
# solution of a student - TODO: adapt to current code - might not be needed seems pretty atomic

# first step: webscraping for the urls of the artists wikipedia entries
# therefore making an adjusted dataframe of the participants-dataframe from part 1 which includes the links to the pages of the respective artist
# using 2010
url_2010 = "https://en.wikipedia.org/wiki/Eurovision_Song_Contest_2010"
resp = requests.get(url_2010)
soup = BeautifulSoup(resp.content, "html.parser")
tables_ = soup.find_all("table", class_ = "wikitable plainrowheaders")

# filtering the tables for the participant table
for table in tables_:
    caption = table.find("caption")
    if caption and "Participants of the Eurovision Song Contest 2010" in caption.text:
            participant_table = table
            break

# filtering the artist-links by looking for "href" within "a" of the participant table (i==2 because the artists are the third element labeled with
# "th" or "td" within each "tr"
rows = []
for row in participant_table.find_all("tr"):
    cells = row.find_all(["th", "td"])
    row_artist_link = None
    for i, cell in enumerate(cells):
        if i == 2:
            url_artist = cell.find("a")
            if url_artist and "href" in url_artist.attrs:
                row_artist_link = url_artist["href"]
    rows.append(row_artist_link)

# delete first element of "rows" because first row/tr has no link (just the headers of the table)
del rows[0]

# creating a variable based on the table of part 1 task 2 (participants) 
df_2010 = part_10 #dataframes_participants[0]

# creating a dictionary with a key and the respective links as values in order to be able to add them to the participants table
dict_urls = {"Artist_urls": rows}

# adding the links to the participants table
for key, value in dict_urls.items():
    df_2010[key] = value

# check
df_2010

# note: for some countries, multiple artists performed at the ESC
# This approach only uses the pages of the first named artist of each country,
# assuming that this was the "main" artist.
# Alternatively, one could also check which artists page was created earlier and use that one

In [ ]:
# getting the meta-information on each artists` wiki page by using page/pageinfo (again with loop and f-string-structure) via API call
# removing the first 6 signs of the Artist_urls (/wiki/) by using replace
responses_info= []
for artist in df_2010["Artist_urls"]:
    artist_page = artist.replace("/wiki/", "")
    url = f"https://xtools.wmcloud.org/api/page/pageinfo/en.wikipedia.org/{artist_page}?format=json"
    responses = requests.get(url)
    responses_info.append(responses.json())

In [ ]:
# checking the format of the creation date
print(responses_info[0]["created_at"])

# making a list of the dates when each page was created (with a loop)
creation_list = []
for artist in responses_info:
    creation_list.append(artist["created_at"])

# removing the last 10 signs so that only the day of the date is left
# therefore using a comprehension
creation_list = [creation_date[:-10] for creation_date in creation_list]

# making a dictionary of theses dates and adding them to the dataframe for better overview
dict_creation = {"creation_date": creation_list}

for key, value in dict_creation.items():
    df_2010[key] = value


# prepare another column with the dates transformed into their "age" (age on the 24th of November 2024)
df_2010["page_age"] = [datetime.strptime("2024-11-24", "%Y-%m-%d") - datetime.strptime(date, "%Y-%m-%d") for date in df_2010["creation_date"]]


df_2010

***b) Investigate if there exists a relationship between the creation date and the voting result.***

Does the hypothesis, that artists whose Wikipedia page is created earlier ahead of the contest recieve higher scores, hold for your year? Create some nice visualization to back your findings!<br>
Did you encounter any problems with your approach? If yes, name them and shortly explain their impact on your analysis.

In [ ]:
# first, merge the two tables on the countries so that the total score and the age of the wiki-page 
# can be compared
# for the inverstigation the total scores are the only relevant information
print(df_2010.columns)
df_2010_reg = pd.merge(df_2010[["Country", "page_age"]], ev_2010[["Country", "Total score"]], on = "Country", how = "inner")

# writing a function that changes "page_age" to a int variable
def days_int(age):
    """
    changing the type of page_age to int
    """
    return int(str(age.days))

# using the function on the page_age column and also changing "total score" into integer values
df_2010_reg["page_age"] = df_2010_reg["page_age"].apply(days_int) 
df_2010_reg["Total score"] =  [int(score) for score in df_2010_reg["Total score"]]
   

df_2010_reg

In [ ]:
# using a linear regression model for investigation

# defining x- and y-values
x = df_2010_reg[["page_age"]]
y = df_2010_reg[["Total score"]]

# model
model = LinearRegression()
model.fit(x, y)

# regression line
reg_model = model.predict(x)

# plotting the data and the regression line with additional information
plt.scatter(x, y, color = "cadetblue", label = "participants")
plt.plot(x, reg_model, color = "turquoise", label = "regression line")
plt.xlabel("age of participants` wiki-page in days")
plt.ylabel("score in ESC final")
plt.title("Relation between ESC participants` score and the age of their wiki-page")
plt.legend()
plt.show()

In [ ]:
# statistics of the linear regression model for more detailed information
x = sm.add_constant(x)

model = sm.OLS(y, x)
results = model.fit()
print(results.summary())

Quite bad statistics, r-squated is only 0,024 (which means that only 2,4% of the data is explained by the model)... Additionally, p-value is extremly high, indicating that the model is not significant and the age of the artists wiki-page does not influence the score in the finale

Even if the model was significant, the coefficient of page_age is 0.0168, meaning that for each day the page was created earlier, the score increases by 0.0168. Following that, for an increase of 10 points the page had to be created 1000 days earlier (around 2,7 years).

***

<h1>Part 3 - Network Visualization</h1>


<h4>Task 4 - Vizualize the results in a network graph.</h4>

Choose one of the years you scraped above (i.e. the year, for which you also got the artist-data) and visualize the voting results in a network plot. Play around with different layouts and mappings (for example: how do you indicate contestants vs. only voters, how could you visualize the overall points a country got, how can you display the assigned points?).

*Hint:* It makes sense to restrict your plot on i.e. only the three highest points (8, 10, 12) to prevent it from becoming too cluttered.

***a) Familiarize with Graphs***

Extract i.e. the entities being mapped to the nodes and the entities being mapped to edges, etc.

In [ ]:
# this might be doubled code - TODO: check

# get the age of every artist's (from the ESC 2010) Wikipedia page no matter whether they are a finalist or not
df_10_testing = part_10.merge(ev_2010, left_on = 'Country', right_on = 'Constestant', how='outer')

In [2]:
# get all nodes

# Initialize a list to store all nodes
vertices = []

# Loop over the df contining all artists/countries (no matter whether a finalist or not) and all points given in the final row by row
for row in range(0, df_10_testing.shape[0]):
    
    # get the country name and store it
    country = df_10_testing['Country'].iloc[row]

    # decided whether the country's artist is a finalist (Contestant) or a non-finalist (only_voters) and store it
    participant = 'only_voters' if pd.isna(df_10_testing['Constestant'].iloc[row]) else 'Contestant'
    
    # Since the minimum value someone got in the semifinals is 2 we will set this as the total score for everyone that did not make it to the finals - finalist get their achieved points
    total_score = 2 if pd.isna(df_10_testing['Total score'].iloc[row]) else  df_10_testing['Total score'].iloc[row]
    
    # add the information gathered above about the country/artist as a node into the verices list inizialist above
    vertices.append((country, {'participant' : participant, 'total_score' : total_score}))
    
vertices 


In [ ]:
# get all edges
# all edges and points are stored in ev_10 for simplicity, this df will be used

# initialze a list to store all the edges
edges = []

# loop over the columns therefore all the countries that can vote
for voter in ev_2010.columns[2:]:
    # for each voter country (only_voters AND contestants) loop over the points they gave to each contestant (finalists)
    for i in range(ev_2010.shape[0]):
        # check whether they gave points to the current contestant
        if (ev_2010[voter].iloc[i] != ''):
            # store the country they gave the points to
            receiver = ev_2010['Constestant'].iloc[i]
            # store the amount of points
            points = ev_2010[voter].iloc[i]
            # append these information as a tuple to the edges list
            edges.append((voter, receiver, points))
        else:
            # if no points have been given to the current contestant from the current voter country, then do nothing and continue with the loop
            continue

edges

In [ ]:
# add nodes and edges to a directed graph

# initialize a directed graph
DG = nx.DiGraph()

# add the nodes prepared before in the vertices list to the directed graph
DG.add_nodes_from(vertices)
# add the edges prepared before in the edges list to the directed graph
DG.add_weighted_edges_from(edges)

In [ ]:
# also create a trimmed version 
# (only with edges where the points given >= 8)
filtered_edges = [(voter, contestant, points) for voter, contestant, points in edges if int(points) >= 8]

# trimmed graph
# repeate the steps from the code chunk before, keeping all the nodes and only keeping edges between two nodes where the points given are 8 or higher
DG_trimmed = nx.DiGraph()
DG_trimmed.add_nodes_from(vertices)
DG_trimmed.add_weighted_edges_from(filtered_edges)

***b) Vizualize your choosen year***


In [ ]:
# Solution - Naive

# visualising the trimmed graph and a circular layout, showing the labels of the nodes
nx.draw(DG_trimmed, pos = nx.circular_layout(DG_trimmed), with_labels = True)

In [ ]:
# Change NODE COLOR dependend on PARTICIPANT TYPE in the finals

# Create a color map based on the 'participant' attribute
color_map = {'Contestant': 'orange', 'only_voters': 'yellow'}

# Extract each node with its participant type & store in a dict - using list comprehension
participant_values = {node: data['participant'] for node, data in DG_trimmed.nodes(data=True)}

# Get the node colors in a list based on the color map and participant type - using list comprehension (and 'nested indexing')
node_colors = [color_map[participant_values[node]] for node in DG_trimmed.nodes]

In [ ]:
# Change SIZE of the NODE depending on the TOTAL SCORE

# As before, extract each node with its total_score value
total_score_values = {node: int(data['total_score']) for node, data in DG_trimmed.nodes(data=True)}

# Get the node size in a list and multiplying them with 3 so the differences are better visible
node_sizes = [total_score_values[node]*3 for node in DG_trimmed.nodes]

In [ ]:
# Change WIDTH of the EDGES depending on the POINTS received
edge_weights = [int(data['weight'])-7 for _, _, data in DG_trimmed.edges(data=True)]

In [ ]:
# Draw the graph with customized node colors, node size and edge size

# increase figure size
plt.figure(figsize=(10, 8)) 

# Draw the graph with customized node color, node size and hiding the edges by coloring them white
nx.draw(DG_trimmed, pos=nx.circular_layout(DG_trimmed), node_color=node_colors, node_size=node_sizes, edge_color = 'white')

# add edges with weight according to the points given in each edge
edges = nx.draw_networkx_edges(DG_trimmed, pos=nx.circular_layout(DG_trimmed), width=edge_weights, edge_color='black', alpha=0.4)

# Add the labels
labels = nx.draw_networkx_labels(DG, pos=nx.circular_layout(DG_trimmed), font_color='black', font_size=8)

# Adjust the labels - going inside the labels 'object' and adjusting some of the positions and adding new lines to long country names
# the values used to adjust had been a trail and error with different values till a satisfactory visualisation was achieved
for label in labels:
    # extracting the text_object from the dictionary
    text_object = labels[label]
    # extracting the position value of x and y
    x, y = text_object.get_position()[0], text_object.get_position()[1]

    # for labels made up of at least two words, write them in two lines
    if ' ' in label:
        new_text = label[::-1].replace(' ', '\n', 1)[::-1] #reversing the string twice to only replace the last white space (not the first)
        text_object.set_text(new_text) #store new lable
    
    # changing the position depending on the position - the idea is to move every lable more outwards
    # so here the labels right and left more right and left respectively
    if (abs(x) > 0.50):
        new_x = x * 1.15 # adjust difference from the original position
        text_object.set_position((new_x, y)) # store new new info 
    # and here the labels on the top and bottom, lower or higher respectively
    elif(abs(y)> 0.8):
        new_y = y * 1.05  # adjust difference from the original position
        text_object.set_position((x, new_y)) #store new info


# add figure infos
plt.title('Voting Results of the ESC 2010')
plt.suptitle('This is a directed graph showing who gave whom points. \n The edge size correlates with the points given and the node size with all points received.', fontsize =8)

# Legend for participant type
# decided which lables should be displyed in the ledgend - using a dictionary to store them
legend_labels = {'Contestant': 'Contestant (finalists)', 'only_voters': 'Voter (non-finalists)'}
# create the color field that is going to be displyed next to the lables
legend_handles = [Patch(color=color_map[label], label=legend_labels[label]) for label in color_map]
# add the legend with the above defined labels and handels
plt.legend(handles=legend_handles, labels=legend_labels.values(), loc='upper right')

***

<h1>Part 4 - Bonus</h1>


Unfortunately, the network plots for single years of the Eurovision Songcontest are not too revealing...

This might change, if we aggregate data from several years. Then, we might be able to actually detect communities of countries within the data. That's what we would like you to investigate now.

In the GitHib folder, we provided you Eurovision vote results from 10 years (2010 - 2020). 

***a) Import the data files***

Import the csv-files and aggregate the data by taking i.e. the sum or average of points a county A (in columns) assigned a country B (the rows) over the years. 



In [ ]:
# again a student solution because the graph looked really good (refers to ALL SUBTASKS)

data = pd.read_csv("data_10_years.csv")
#print(data[data["Contestants"] == "Spain"])
#print(data[data["Contestants"] == "Spain"]["Romania"].mean())

# summing up the countries` values by setting a not-in if condition  
agg_data_ = {}
for column in data.columns:
    if column not in ["Unnamed: 0", "Unnamed: 1", "Contestants", "Total score", "Jury score", "Televoting score"]:
        agg_data_[column] = "sum"

# aggregating the years of each country
agg_data = data.groupby(["Contestants"]).agg(agg_data_).reset_index()
agg_data

# not all countries that are in the columns have their own row -> same procedure as above
# list with contries that aren`t in the rows but in the columns 
columns_yes_rows_no_ = set(agg_data.columns[2:]) - set(agg_data["Contestants"])
columns_yes_rows_no_

# add countries as rows with zeros (just as above)
for country in columns_yes_rows_no_:
    row = [country] + [0] * len(agg_data.columns[1:].tolist())
    agg_data.loc[len(agg_data)] = row
agg_data

# to me, it seemed a bit weird that Slovakia never participated in the finale over the whole period of
# time, but a quick search on Wikipedia confirms this (Slovakia is the only country that is added as a row)

# creating the adjacency_matrix
adjacency_matrix_agg = agg_data.set_index("Contestants")[agg_data.columns[1:]].astype(int)

# set all values below 60 to zero (threshold so that the plot stays readable)
for index, row in adjacency_matrix_agg.iterrows():
    for cell in adjacency_matrix_agg.columns:
        adjacency_matrix_agg[adjacency_matrix_agg < 60] = 0


adjacency_matrix_agg

***b) Visualize the result***

Visualize these aggregated scores in a network plot. The layout type `kamada_kawai_layout` is suitable for detecting communities. Make sure to weight edges by the assigned number of points.

In [ ]:
# make a network
agg_network = nx.from_pandas_adjacency(adjacency_matrix_agg, create_using = nx.DiGraph)

# make sure only those countries with a relevant connection are used in the network plot (those that lack any connections are filtered by if condition)
nodes_cleared = [node for node, edges in dict(agg_network.degree()).items() if edges > 0]
cleared_network = agg_network.subgraph(nodes_cleared)

# reverse the network since networkx uses the rows as "from" and the columns as "to" but the matrix is built in the reverse way
reversed_network = cleared_network.reverse()

# inverted weights; I don`t know why, but without that step it seems to me like the countries are 
# further apart if they actually have a stronger connection according to the points in the matrix, so I
# inverted the weights for every edge
inverted_weights = {
    (x, y): 1 / d["weight"] if d["weight"] != 0 else 0   # make sure there is no division by 0
    for x, y, d in reversed_network.edges(data=True)
}

nx.set_edge_attributes(reversed_network, inverted_weights, "weight")

layout = nx.kamada_kawai_layout(reversed_network, weight = "weight")

# I know, this is bad practice, but some nodes are overlapping (especially those that are not directly connected but share the same edges to a common
# third node (Ireland, UK and Hungary to Sweden; Cyprus & France). I didn`t find a easily implementable solution, so I most slightly changed the 
# coordinates manually 
plt.figure(figsize=(9, 7))
layout["Ireland"] = (-0.70041447, 0.46745569)
layout["Hungary"] = (-0.75551272, 0.34775535)
layout["Cyprus"] = ( 0.04136158, -0.06362613)

nx.draw(
    reversed_network,
    pos = layout,
    edge_color= "firebrick",
    with_labels=True,
    node_color = "linen",
    font_size = 8,
    node_size = 1100
    )

plt.title("Connections between ESC contestants 2010-2019")
plt.show()

***c) Interprete your findings***

Interprete your plot. Could you detect something?

Generally speaking, there seems to be a relation between geographical and cultural neighbourhoods and the support in the ESC finals. For example, Italy seems to enjoy a lot of support from its (indirect) neighbours like Malta, Switzerland and San Marino as well as from historically connected countries (Malta again, Albania).
Another good example for such historically, culturally or geographically formed alliances would be former members of the Soviet Union (Russia, Belarus, Moldowa, Azerbaijan etc.)

Some thoughts on the procedure: this approch only uses the sum of the points of the aggregated years. This means that the approach neglects the number of years a country has participated in the finale (obviously, a country that has only participated twice in the finale over the ten years can`t receive more than 60 points from another country. Still, if a country only participated twice, having received many points by another country (for example more than 20 points) may just be result by chance or because of the songs rather than by structural connections between countries. Following this thought, it seems reasonable to set the threshold at 60 points (half of the possible points assignable over the period of 10 years).